# Tile and LULC Clipping Investigation

Visual-only notebook for diagnosing why part of the country appears clipped in field-change outputs.

The notebook overlays:
- clipped mosaic tile bounds from `02_clipped_mosaics`,
- LULC raster coverage from `03_lclu_masks`,
- per-tile vector export bounds from `07_exports`,
- merged field vector bounds from `08_instance_postprocess*`,
- optional country boundary.

No change metrics are computed here. Use the plots to identify the first pipeline stage where coverage becomes incomplete.

In [ ]:
from pathlib import Path
import math
import re
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.plot import show as rio_show
from shapely.geometry import box
from IPython.display import display

## Configuration

Adjust only this cell for a different machine, country boundary, or year range.

In [ ]:
RUNS_ROOT = Path('/mnt/KSA-Oasis/houcine/field_delineation_data/field_delineation_runs')
YEARS = list(range(2020, 2026))
SEASONS = ['february_april', 'june_august']

FILENAME_TEMPLATES = [
    'kursh_{year}_images_season_{season}__dw_lulc',
    'kursh_{year}_{season}__dw_lulc',
    'kursh_{year}_{season}',
]

# Optional. Set to a GeoJSON/GPKG boundary if you want country outline overlays.
COUNTRY_BOUNDARY_PATH = None

# Visual settings.
TARGET_CRS = 'EPSG:32636'
SAVE_FIGURES = False
FIGURE_OUTPUT_DIR = Path('/mnt/noobjam/change_detection_fields_output/diagnostics/tile_lulc_clipping')
RASTER_PREVIEW_MAX_SIZE = 1600
MAX_LULC_TILES_PER_PLOT = 16

# Optional: draw actual merged field polygons for a selected snapshot.
# This can be slow for country-scale GeoJSONs, so footprint bounds are used by default.
PLOT_FULL_MERGED_FIELDS = False
MAX_MERGED_FIELD_FEATURES = 75000

## Discovery Helpers

These helpers locate snapshot folders and collect the visual layers for each snapshot.

In [ ]:
TILE_RE = re.compile(r'\b\d{2}[A-Z]{3}\b')


def unique_paths(paths):
    seen = set()
    unique = []
    for path in paths:
        resolved = str(path)
        if resolved not in seen:
            unique.append(path)
            seen.add(resolved)
    return unique


def snapshot_label(year, season):
    return f'{year}_{season}'


def guess_tile_id(path):
    for part in reversed(path.parts):
        match = TILE_RE.search(part)
        if match:
            return match.group(0)
    return path.stem


def discover_snapshot_runs():
    snapshots = {}
    missing = []
    ambiguous = []
    for year in YEARS:
        for season in SEASONS:
            label = snapshot_label(year, season)
            selected = None
            for template in FILENAME_TEMPLATES:
                pattern = template.format(year=year, season=season)
                exact = RUNS_ROOT / pattern
                candidates = []
                if exact.exists():
                    candidates.append(exact)
                candidates.extend(path for path in sorted(RUNS_ROOT.glob(pattern)) if path.exists())
                candidates = unique_paths(candidates)
                if candidates:
                    selected = candidates[0]
                    if len(candidates) > 1:
                        ambiguous.append((label, candidates))
                    break
            if selected is None:
                missing.append(label)
            else:
                snapshots[label] = {
                    'label': label,
                    'year': year,
                    'season': season,
                    'run_dir': selected,
                }
    if missing:
        print('Missing run folders:')
        for label in missing:
            print(f'  - {label}')
    if ambiguous:
        print('Ambiguous run folders, using the first candidate for each:')
        for label, candidates in ambiguous:
            print(f'  - {label}: {candidates[0]}')
    return snapshots


def collect_snapshot_paths(run_dir):
    clipped_mosaics = unique_paths(
        sorted(run_dir.glob('02_clipped_mosaics/*.tif'))
        + sorted(run_dir.glob('02_clipped_mosaics/*.tiff'))
        + sorted(run_dir.glob('02_clipped_mosaics/*.vrt'))
    )
    lulc = unique_paths(
        sorted(run_dir.glob('03_lclu_masks/**/*.tif'))
        + sorted(run_dir.glob('03_lclu_masks/**/*.tiff'))
        + sorted(run_dir.glob('03_lclu_masks/**/*.vrt'))
    )
    tile_vectors = unique_paths(
        sorted(run_dir.glob('07_exports/*/*/*.geojson'))
        + sorted(run_dir.glob('07_exports/*/*.geojson'))
        + sorted(run_dir.glob('07_exports/**/*.gpkg'))
    )
    merged_vectors = unique_paths(
        sorted(run_dir.glob('08_instance_postprocess*/*.merged_fields.geojson'))
        + sorted(run_dir.glob('08_instance_postprocess*/*.merged_fields.gpkg'))
    )
    return {
        'clipped_mosaics': clipped_mosaics,
        'lulc': lulc,
        'tile_vectors': tile_vectors,
        'merged_vectors': merged_vectors,
    }


def load_country_boundary(target_crs):
    if COUNTRY_BOUNDARY_PATH is None:
        return None
    boundary_path = Path(COUNTRY_BOUNDARY_PATH)
    if not boundary_path.exists():
        warnings.warn(f'Country boundary path does not exist: {boundary_path}')
        return None
    boundary = gpd.read_file(boundary_path)
    if boundary.crs is None:
        warnings.warn('Country boundary has no CRS. Assigning EPSG:4326 by default.')
        boundary = boundary.set_crs('EPSG:4326')
    return boundary.to_crs(target_crs)

## Footprint Helpers

These functions plot extents only. They are fast enough to run across many snapshots and are the first diagnostic for clipping.

In [ ]:
def raster_bounds_gdf(paths, layer_name, target_crs=TARGET_CRS):
    frames = []
    for path in paths:
        try:
            with rasterio.open(path) as src:
                if src.crs is None:
                    warnings.warn(f'Raster has no CRS and will be skipped: {path}')
                    continue
                geom = box(src.bounds.left, src.bounds.bottom, src.bounds.right, src.bounds.top)
                gdf = gpd.GeoDataFrame(
                    {
                        'layer': [layer_name],
                        'tile_id': [guess_tile_id(path)],
                        'path': [str(path)],
                    },
                    geometry=[geom],
                    crs=src.crs,
                ).to_crs(target_crs)
                frames.append(gdf)
        except Exception as exc:
            warnings.warn(f'Could not read raster bounds for {path}: {exc}')
    if not frames:
        return gpd.GeoDataFrame(columns=['layer', 'tile_id', 'path', 'geometry'], geometry='geometry', crs=target_crs)
    return gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), geometry='geometry', crs=target_crs)


def vector_total_bounds(path):
    try:
        import pyogrio
        info = pyogrio.read_info(path)
        bounds = info.get('total_bounds')
        crs = info.get('crs') or info.get('crs_wkt')
        if bounds is not None and len(bounds) == 4:
            return bounds, crs
    except Exception:
        pass
    gdf = gpd.read_file(path)
    return gdf.total_bounds, gdf.crs


def vector_bounds_gdf(paths, layer_name, target_crs=TARGET_CRS):
    frames = []
    for path in paths:
        try:
            bounds, crs = vector_total_bounds(path)
            if crs is None:
                warnings.warn(f'Vector has no CRS and will be treated as EPSG:4326: {path}')
                crs = 'EPSG:4326'
            geom = box(float(bounds[0]), float(bounds[1]), float(bounds[2]), float(bounds[3]))
            gdf = gpd.GeoDataFrame(
                {
                    'layer': [layer_name],
                    'tile_id': [guess_tile_id(path)],
                    'path': [str(path)],
                },
                geometry=[geom],
                crs=crs,
            ).to_crs(target_crs)
            frames.append(gdf)
        except Exception as exc:
            warnings.warn(f'Could not read vector bounds for {path}: {exc}')
    if not frames:
        return gpd.GeoDataFrame(columns=['layer', 'tile_id', 'path', 'geometry'], geometry='geometry', crs=target_crs)
    return gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), geometry='geometry', crs=target_crs)


def footprint_layers_for_snapshot(snapshot, target_crs=TARGET_CRS):
    paths = collect_snapshot_paths(snapshot['run_dir'])
    return {
        'clipped_mosaics': raster_bounds_gdf(paths['clipped_mosaics'], 'clipped mosaics', target_crs),
        'lulc': raster_bounds_gdf(paths['lulc'], 'LULC masks', target_crs),
        'tile_vectors': vector_bounds_gdf(paths['tile_vectors'], 'tile vector exports', target_crs),
        'merged_vectors': vector_bounds_gdf(paths['merged_vectors'], 'merged field vectors', target_crs),
    }


LAYER_STYLES = {
    'clipped_mosaics': {'edgecolor': '#2563eb', 'linewidth': 1.4, 'linestyle': '-', 'label': 'clipped mosaic bounds'},
    'lulc': {'edgecolor': '#16a34a', 'linewidth': 1.4, 'linestyle': '--', 'label': 'LULC raster bounds'},
    'tile_vectors': {'edgecolor': '#f97316', 'linewidth': 1.2, 'linestyle': ':', 'label': 'per-tile export bounds'},
    'merged_vectors': {'edgecolor': '#dc2626', 'linewidth': 2.1, 'linestyle': '-', 'label': 'merged field bounds'},
}


def plot_layer_outline(ax, gdf, style):
    if gdf.empty:
        return
    gdf.boundary.plot(
        ax=ax,
        color=style['edgecolor'],
        linewidth=style['linewidth'],
        linestyle=style['linestyle'],
        label=style['label'],
    )


def label_tiles(ax, gdf, color='#111827'):
    if gdf.empty:
        return
    for _, row in gdf.iterrows():
        point = row.geometry.representative_point()
        ax.text(
            point.x,
            point.y,
            str(row['tile_id']),
            fontsize=8,
            color=color,
            ha='center',
            va='center',
            bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.65, 'pad': 1.5},
        )


def apply_equal_view(ax):
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, linewidth=0.3, alpha=0.35)
    ax.set_xlabel('x')
    ax.set_ylabel('y')


def save_figure_if_enabled(fig, name):
    if not SAVE_FIGURES:
        return
    FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURE_OUTPUT_DIR / name
    fig.savefig(path, dpi=180, bbox_inches='tight')
    print(f'Saved figure: {path}')

## Snapshot Footprint Plot

Use this plot first. If the clipped part is already missing in blue or green, the issue is upstream of vector export.
If blue/green are complete but orange/red are clipped, the issue is in export or merge.

In [ ]:
def plot_snapshot_footprints(label, target_crs=TARGET_CRS, ax=None, show_legend=True, show_title=True):
    snapshot = SNAPSHOTS[label]
    layers = footprint_layers_for_snapshot(snapshot, target_crs)
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 12))
    else:
        fig = ax.figure

    country = load_country_boundary(target_crs)
    if country is not None:
        country.boundary.plot(ax=ax, color='black', linewidth=1.3, label='country boundary')

    for layer_name in ['clipped_mosaics', 'lulc', 'tile_vectors', 'merged_vectors']:
        plot_layer_outline(ax, layers[layer_name], LAYER_STYLES[layer_name])

    label_source = layers['tile_vectors']
    if label_source.empty:
        label_source = layers['clipped_mosaics']
    label_tiles(ax, label_source)

    apply_equal_view(ax)
    if show_title:
        ax.set_title(f'{label}: tile, LULC, export, and merged-vector coverage')
    if show_legend:
        ax.legend(loc='best')
    save_figure_if_enabled(fig, f'{label}_footprints.png')
    return fig, ax

## Season Footprint Grids

These grids compare the same season across years. Use them to see if the clipped side appears in one year only or across the whole time series.

In [ ]:
def labels_for_season(season):
    return [label for label, snapshot in sorted(SNAPSHOTS.items(), key=lambda item: item[1]['year']) if snapshot['season'] == season]


def plot_season_footprint_grid(season, target_crs=TARGET_CRS):
    labels = labels_for_season(season)
    if not labels:
        raise ValueError(f'No snapshots found for season: {season}')
    cols = 3
    rows = math.ceil(len(labels) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 7))
    axes = np.array(axes).reshape(-1)
    for ax, label in zip(axes, labels):
        plot_snapshot_footprints(label, target_crs=target_crs, ax=ax, show_legend=False, show_title=True)
    for ax in axes[len(labels):]:
        ax.axis('off')
    handles, legend_labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, legend_labels, loc='lower center', ncol=4)
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    save_figure_if_enabled(fig, f'{season}_footprint_grid.png')
    return fig

## LULC Raster Preview Helpers

These plots draw downsampled LULC rasters and overlay the other coverage layers. They are visual previews, not measurements.

In [ ]:
def raster_preview(path, max_size=RASTER_PREVIEW_MAX_SIZE):
    with rasterio.open(path) as src:
        scale = max(src.width / max_size, src.height / max_size, 1.0)
        out_height = max(1, int(src.height / scale))
        out_width = max(1, int(src.width / scale))
        array = src.read(1, out_shape=(out_height, out_width), masked=True)
        transform = src.transform * src.transform.scale(src.width / out_width, src.height / out_height)
        return array, transform, src.crs


def raster_crs(path):
    with rasterio.open(path) as src:
        return src.crs


def plot_lulc_snapshot(label, max_tiles=MAX_LULC_TILES_PER_PLOT):
    snapshot = SNAPSHOTS[label]
    paths = collect_snapshot_paths(snapshot['run_dir'])
    lulc_paths = paths['lulc'][:max_tiles]
    if not lulc_paths:
        raise FileNotFoundError(f'No LULC rasters found for {label}')

    plot_crs = raster_crs(lulc_paths[0]) or TARGET_CRS
    fig, ax = plt.subplots(figsize=(11, 13))

    for path in lulc_paths:
        try:
            array, transform, crs = raster_preview(path)
            if crs != plot_crs:
                warnings.warn(f'Skipping LULC raster with different CRS: {path}')
                continue
            rio_show(array, transform=transform, ax=ax, cmap='tab20', alpha=0.72, interpolation='nearest')
        except Exception as exc:
            warnings.warn(f'Could not plot LULC preview for {path}: {exc}')

    layers = footprint_layers_for_snapshot(snapshot, str(plot_crs))
    for layer_name in ['clipped_mosaics', 'tile_vectors', 'merged_vectors']:
        plot_layer_outline(ax, layers[layer_name], LAYER_STYLES[layer_name])
    label_source = layers['tile_vectors']
    if label_source.empty:
        label_source = layers['clipped_mosaics']
    label_tiles(ax, label_source)

    if PLOT_FULL_MERGED_FIELDS and paths['merged_vectors']:
        merged = gpd.read_file(paths['merged_vectors'][0], rows=slice(0, MAX_MERGED_FIELD_FEATURES))
        if merged.crs is None:
            merged = merged.set_crs('EPSG:4326')
        merged.to_crs(plot_crs).boundary.plot(ax=ax, color='#111827', linewidth=0.15, alpha=0.35)

    apply_equal_view(ax)
    ax.set_title(f'{label}: LULC preview with tile/export/merged coverage overlays')
    ax.legend(loc='best')
    save_figure_if_enabled(fig, f'{label}_lulc_preview.png')
    return fig, ax


def plot_lulc_grid_for_season(season, max_tiles=MAX_LULC_TILES_PER_PLOT):
    labels = labels_for_season(season)
    cols = 2
    rows = math.ceil(len(labels) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 8, rows * 9))
    axes = np.array(axes).reshape(-1)
    for ax, label in zip(axes, labels):
        snapshot = SNAPSHOTS[label]
        paths = collect_snapshot_paths(snapshot['run_dir'])
        lulc_paths = paths['lulc'][:max_tiles]
        if not lulc_paths:
            ax.set_title(f'{label}: no LULC rasters found')
            ax.axis('off')
            continue
        plot_crs = raster_crs(lulc_paths[0]) or TARGET_CRS
        for path in lulc_paths:
            try:
                array, transform, crs = raster_preview(path)
                if crs != plot_crs:
                    continue
                rio_show(array, transform=transform, ax=ax, cmap='tab20', alpha=0.72, interpolation='nearest')
            except Exception as exc:
                warnings.warn(f'Could not plot LULC preview for {path}: {exc}')
        layers = footprint_layers_for_snapshot(snapshot, str(plot_crs))
        for layer_name in ['clipped_mosaics', 'tile_vectors', 'merged_vectors']:
            plot_layer_outline(ax, layers[layer_name], LAYER_STYLES[layer_name])
        label_source = layers['tile_vectors']
        if label_source.empty:
            label_source = layers['clipped_mosaics']
        label_tiles(ax, label_source)
        apply_equal_view(ax)
        ax.set_title(label)
    for ax in axes[len(labels):]:
        ax.axis('off')
    fig.tight_layout()
    save_figure_if_enabled(fig, f'{season}_lulc_grid.png')
    return fig

## Discover Snapshots

Run this cell first. It prints only which snapshot folders will be visualized.

In [ ]:
SNAPSHOTS = discover_snapshot_runs()
for label, snapshot in sorted(SNAPSHOTS.items(), key=lambda item: (item[1]['year'], item[1]['season'])):
    print(f"{label}: {snapshot['run_dir']}")

## Visual 1: Focus on the Suspect Snapshot

Start with the corrected snapshot that was previously problematic.

In [ ]:
plot_snapshot_footprints('2024_june_august')
plt.show()

## Visual 2: Same-Season Footprints Across Years

These reveal whether the clipped side is isolated to one snapshot or inherited across years.

In [ ]:
plot_season_footprint_grid('june_august')
plt.show()

In [ ]:
plot_season_footprint_grid('february_april')
plt.show()

## Visual 3: LULC Preview for the Suspect Snapshot

If the LULC itself is clipped or shifted here, the issue is upstream of field polygon comparison.

In [ ]:
plot_lulc_snapshot('2024_june_august')
plt.show()

## Visual 4: LULC Preview Across Same Season

Use this if the single snapshot plot suggests the LULC is the source of clipping.

In [ ]:
plot_lulc_grid_for_season('june_august')
plt.show()

## Optional: Save Figures

Set `SAVE_FIGURES = True` in the configuration cell, then rerun the visual cells. Files will be written to `FIGURE_OUTPUT_DIR`.